### import essential libraries

In [1]:
from operator import mod
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import RandomOverSampler
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

### Load Data

In [2]:
mnist = fetch_openml('mnist_784', parser='auto')

In [3]:
X, y = mnist['data'], mnist['target'].ravel().astype(np.int64)

### Preprocessing

In [4]:
# Normalize the data
X = X / 255.0

In [5]:
# Standardize the data
scaler = StandardScaler()
X = scaler.fit_transform(X)

### train- test split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Define a function to imbalance the data

In [7]:
def create_imbalanced_dataset(X, y, ratios):
    unique_classes = np.unique(y)
    if len(unique_classes) < 2:
        raise ValueError('The dataset must contain at least two classes.')

    X_imbalanced, y_imbalanced = [], []
    for i, cls in enumerate(unique_classes):
        samples = X[y == cls]
        n_samples = min(max(int(len(X) * ratios[i] / sum(ratios)), 1), len(samples))

        resampled_samples = resample(samples, replace=False, n_samples=n_samples, random_state=42)

        X_imbalanced.append(resampled_samples)
        y_imbalanced.extend([cls] * n_samples)

    X_imbalanced = np.vstack(X_imbalanced)
    y_imbalanced = np.array(y_imbalanced)

    return X_imbalanced, y_imbalanced

### Define imbalance ratios

In [8]:
imbalance_ratios = [
    (50, 20, 10, 5, 5, 3, 3, 2, 1, 1),
    (45, 25, 10, 5, 5, 3, 3, 2, 1, 1),
    (40, 30, 10, 5, 5, 3, 3, 2, 1, 1),
    (35, 35, 10, 5, 5, 3, 3, 2, 1, 1),
    (30, 40, 10, 5, 5, 3, 3, 2, 1, 1)
]

### for each ratio implement three ideas and evaluate them

In [ ]:
for ratio in imbalance_ratios:
    
    # Create imbalanced training set
    X_train_imbalanced, y_train_imbalanced = create_imbalanced_dataset(X_train, y_train, ratio)

    # Train and evaluate logistic regression on imbalanced dataset
    clf_imbalanced = LogisticRegression(max_iter=1000, multi_class='auto', solver='saga', random_state=42)
    clf_imbalanced.fit(X_train_imbalanced, y_train_imbalanced)
    y_pred_imbalanced = clf_imbalanced.predict(X_test)
    
    # Apply RandomOverSampler to balance the dataset by oversampling the minority classes
    random_over_sampler = RandomOverSampler(sampling_strategy='auto', random_state=42)
    X_train_balanced_with_ros, y_train_balanced_with_ros = random_over_sampler.fit_resample(X_train_imbalanced, y_train_imbalanced)

    # Train and evaluate logistic regression on balanced dataset (with random over sampling)
    model = LogisticRegression(max_iter=1000, multi_class='auto', solver='saga', random_state=42)
    model.fit(X_train_balanced_with_ros, y_train_balanced_with_ros)
    y_pred_balanced_with_ros = model.predict(X_test)

    # Create and train the logistic regression model with class weighting
    model_with_class_weighting = LogisticRegression(class_weight='balanced', random_state=42, multi_class='auto', solver='saga')
    model_with_class_weighting.fit(X_train_imbalanced, y_train_imbalanced)
    y_pred_balanced_with_class_weighting = model_with_class_weighting.predict(X_test)

    # Train and evaluate XGBClassifier with logistic regression as the base learner
    xgb_clf = XGBClassifier(booster='gblinear', objective='multi:softmax', num_class=len(np.unique(y_train_imbalanced)), n_estimators=100, learning_rate=0.1, random_state=42)
    xgb_clf.fit(X_train_imbalanced, y_train_imbalanced)
    y_pred_balanced_with_xgb = xgb_clf.predict(X_test)

    # Calculate performance metrics
    metrics_imbalanced = [accuracy_score(y_test, y_pred_imbalanced), f1_score(y_test, y_pred_imbalanced, average='weighted')]
    metrics_balanced_with_ros = [accuracy_score(y_test, y_pred_balanced_with_ros), f1_score(y_test, y_pred_balanced_with_ros, average='weighted')]
    metrics_balanced_with_class_weighting = [accuracy_score(y_test, y_pred_balanced_with_class_weighting), f1_score(y_test, y_pred_balanced_with_class_weighting, average='weighted')]
    metrics_balanced_with_xgb = [accuracy_score(y_test, y_pred_balanced_with_xgb), f1_score(y_test, y_pred_balanced_with_xgb, average='weighted')]

    print(f"Imbalance ratio: {ratio}")
    print("Imbalanced dataset metrics: Accuracy: {:.4f}, F1-score: {:.4f}".format(*metrics_imbalanced))
    print("Balanced dataset with random over sampling metrics: Accuracy: {:.4f}, F1-score: {:.4f}".format(*metrics_balanced_with_ros))
    print("Balanced dataset with class weighting metrics: Accuracy: {:.4f}, F1-score: {:.4f}".format(*metrics_balanced_with_class_weighting))
    print("Balanced dataset with xgb metrics: Accuracy: {:.4f}, F1-score: {:.4f}".format(*metrics_balanced_with_xgb))
    print("\n")

/usr/local/lib/python3.9/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.9/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.9/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
